# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Answer

**Base grain** (what the warehouse table actually stores): one row in `fact_content_daily_performance` = one `(report_date, client_hash_id, content_hash_id)` daily performance record. Verified with Query 1 (the grain check) below.

**Feature/label grain** (what my lane's model actually needs): one row = one content item, aggregated over a calendar month, for one `client_hash_id` + `content_hash_id` pair. I build this by summing/averaging the daily rows for March per content item.

**Time window:**
- Feature window: `report_date` in `2026-03-01` → `2026-03-31` (the `month=2026-03` partition) — the mid-panel month I build features from.
- Comparison window, for the proxy label only, never a feature: `report_date` in `2026-02-01` → `2026-02-28` (`month=2026-02`) — last month's total impressions, used only to compute the percent change that defines "declining."
- Everything from `2026-04-01` onward is future information relative to the March decision point and is off-limits (that's the sealed final month territory the assignment warns about).

**Table(s) I use, and why:**
- `fact_content_daily_performance` — primary source for observable signals (GSC impressions/clicks/position, GA4 pageviews/engagement), read one `month=` partition at a time.
- `dim_content` — joined in for content-level static fields (`content_type`, `word_count`, `backlinks`, `content_created_date`) needed for the content-age feature and to filter to published, non-deleted pages.
- `dim_clients` — light context only (`has_gsc_access` sanity check), not a feature source.
- `fact_content_query_90d` — excluded from this exercise. Its window is a fixed `window_start`/`window_end`, not a `month=` partition, and it overlaps the sealed final months per the panel warning. Pulling `impressions_last30`/`impressions_prev30` from it to build a March label would leak information from months I'm not supposed to be using yet.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Answer

**Feature** (knowable before the March decision moment, safe to use):
- `gsc_impressions`, `gsc_clicks`, `gsc_avg_position` — aggregated over March, from `fact_content_daily_performance`
- `ga4_pageviews`, `ga4_engaged_sessions`, `ga4_total_engagement_sec` — aggregated over March, only for rows where `ga4_data_available IS TRUE`
- `word_count`, `content_type`, `backlinks`, `content_created_date` (→ derived content-age) — from `dim_content`

**Label / proxy** (the thing I predict, or what it's computed from — never a feature):
- `is_declining_proxy` = 1 if a content item's total March GSC impressions fell more than 20% versus its total February impressions, else 0. This mirrors the `trend_direction` definition from w02, but computed from two monthly aggregates of the daily fact table rather than from `fact_content_query_90d`'s pre-built `last30`/`prev30` columns, precisely to keep it inside the March window and out of the sealed final months. This label covers only the refresh/prune branch of Lane 2's 5-way target — see the scope note in Section 1.
- `gsc_impressions_feb` (the prior-month total used to build the label) and any percent-change-from-Feb-to-March quantity are label-derived — they must never appear as model features. Section 3's trap exists to prove why.

**Context** (for grouping/joining/reading, never for the model to learn from):
- `client_hash_id`, `content_hash_id` — join/group keys only
- `report_date` — used to build the monthly aggregate windows, not fed to the model as a raw date
- `has_gsc_access`, `gsc_data_start` (from `dim_clients`) — used to sanity-check data availability, not a signal about content quality

**Excluded** (with a one-line why each):
- `fact_content_query_90d` (entire table) — fixed 90-day window misaligned with `month=2026-03`; overlaps sealed final months (leakage risk).
- Any `report_date` in April 2026 or later — future information relative to the March decision point.
- `ga4_*` columns where `ga4_data_available IS FALSE` — these are zero-filled placeholders for clients before their `ga4_data_start`, not genuine zero engagement.
- `keyword_hash_id`, `url_hash_id`, `provider_used`, `model_used` (from `dim_content`) — production/pipeline metadata about how the content was generated, not an organic search-quality signal an editor would reason about.
- `client_created_date`, `access_profile`, `is_active` (from `dim_clients`) — account/business administration fields, not content signals.
- Content rows where `is_deleted IS TRUE` or `is_published IS FALSE` — these aren't live review candidates at all, so they're dropped from the population, not just the feature set.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### 3.0 Setup — connect to the warehouse

Run this first. In Colab: store your token as a Secret named `HF_TOKEN` (key icon in the left sidebar) — never paste it directly into a cell, this repo is public.

In [2]:

%pip install -q duckdb


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: C:\Users\Mithun\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [1]:
# !pip install -q duckdb
import os
import getpass
import duckdb

# Read Hugging Face token securely
HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass(
    "Paste your Hugging Face READ token (hf_...): "
)

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

BASE = "hf://datasets/FlyRank/internship-warehouse"

# Assignment windows
SNAPSHOT = "2026-03-31"   # Decision date
MONTH = "2026-03"         # Primary partition
PREV_MONTH = "2026-02"    # For 60-day trend split

print("✓ Connected to Hugging Face warehouse")
print(f"Snapshot date     : {SNAPSHOT}")
print(f"Primary month     : {MONTH}")
print(f"Comparison month  : {PREV_MONTH}")

✓ Connected to Hugging Face warehouse
Snapshot date     : 2026-03-31
Primary month     : 2026-03
Comparison month  : 2026-02


### 3.1 Query 1 — Grain check

Claim: one row in `fact_content_daily_performance` = one (report_date, client, content) record. If the grain holds, grouping by those three columns and filtering to `COUNT(*) > 1` should return **zero rows**.

In [2]:
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet')
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print(f'Rows violating the claimed grain (should be 0): {len(grain_check)}')
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows violating the claimed grain (should be 0): 0


,report_date,client_hash_id,content_hash_id,n


### 3.2 Query 2 — Row count and date span

Claim: `month=2026-03` covers the full calendar month

In [3]:
count_span = con.sql(f"""
    SELECT
        COUNT(*)                       AS row_count,
        MIN(report_date)               AS min_date,
        MAX(report_date)               AS max_date,
        COUNT(DISTINCT client_hash_id)  AS n_clients,
        COUNT(DISTINCT content_hash_id) AS n_content
    FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet')
""").df()

count_span

,row_count,min_date,max_date,n_clients,n_content
0,9841378,2026-03-01,2026-03-31,55,331437


### 3.3 Query 3 — Availability, filtered with `IS TRUE`

Claim: GA4 columns are zero-filled (not genuinely zero) for rows before a client's `ga4_data_start`, flagged by `ga4_data_available`. Filtering with `IS TRUE` shows how much of the month actually has usable GA4 vs. GSC data.

In [4]:
availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows,
        ROUND(100.0 * SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_gsc_available,
        ROUND(100.0 * SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_ga4_available
    FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet')
""").df()

availability

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows,ga4_available_rows,pct_gsc_available,pct_ga4_available
0,9841378,3611061.0,413966.0,36.7,4.2


### 3.4 Build the March feature frame + the proxy label

One row = one content item, aggregated over March, joined to `dim_content` for static metadata, joined to a February aggregate only to compute the proxy label.

In [5]:
# March aggregate: this month's observable signals, per content item
con.execute(f"""
    CREATE OR REPLACE TABLE march_agg AS
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions)          AS gsc_impressions_mar,
        SUM(gsc_clicks)                AS gsc_clicks_mar,
        AVG(gsc_avg_position)          AS avg_position_mar,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_pageviews ELSE NULL END)         AS ga4_pageviews_mar,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_engaged_sessions ELSE NULL END)  AS ga4_engaged_sessions_mar
    FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""")

# February aggregate: comparison window, used ONLY to build the proxy label
con.execute(f"""
    CREATE OR REPLACE TABLE feb_agg AS
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS gsc_impressions_feb
    FROM read_parquet('{BASE}/fact_content_daily_performance/month={PREV_MONTH}/*.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""")

# Content metadata: static fields, one row per content item
con.execute(f"""
    CREATE OR REPLACE TABLE content_meta AS
    SELECT content_hash_id, client_hash_id, content_type, word_count, backlinks, content_created_date
    FROM read_parquet('{BASE}/dim_content.parquet')
    WHERE is_published IS TRUE AND is_deleted IS FALSE
""")

print('march_agg, feb_agg, content_meta built.')
con.sql('SELECT COUNT(*) AS march_rows FROM march_agg').show()
con.sql('SELECT COUNT(*) AS feb_rows FROM feb_agg').show()
con.sql('SELECT COUNT(*) AS content_rows FROM content_meta').show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

march_agg, feb_agg, content_meta built.
┌────────────┐
│ march_rows │
│   int64    │
├────────────┤
│     176738 │
└────────────┘

┌──────────┐
│ feb_rows │
│  int64   │
├──────────┤
│   153559 │
└──────────┘

┌──────────────┐
│ content_rows │
│    int64     │
├──────────────┤
│       411540 │
└──────────────┘



In [6]:
# Join everything into one content-item-per-row feature frame with the proxy label
feature_frame = con.sql(f"""
    SELECT
        m.client_hash_id,
        m.content_hash_id,
        m.gsc_impressions_mar,
        m.gsc_clicks_mar,
        m.avg_position_mar,
        m.ga4_pageviews_mar,
        m.ga4_engaged_sessions_mar,
        c.content_created_date,
        c.word_count,
        c.content_type,
        c.backlinks,
        f.gsc_impressions_feb,
        -- proxy label: impressions dropped >20% March vs February
        CASE
            WHEN f.gsc_impressions_feb > 0
             AND (m.gsc_impressions_mar - f.gsc_impressions_feb) * 1.0 / f.gsc_impressions_feb <= -0.20
            THEN 1 ELSE 0
        END AS is_declining_proxy,
        -- LEAKAGE TRAP, built on purpose -- do not use as a real feature, see 3.6
        CASE
            WHEN f.gsc_impressions_feb > 0
            THEN (m.gsc_impressions_mar - f.gsc_impressions_feb) * 1.0 / f.gsc_impressions_feb
            ELSE NULL
        END AS pct_change_mar_vs_feb
    FROM march_agg m
    JOIN feb_agg f USING (client_hash_id, content_hash_id)
    JOIN content_meta c USING (client_hash_id, content_hash_id)
    WHERE f.gsc_impressions_feb > 0   -- can't define % change off a zero base
""").df()

print(f'Feature frame rows: {len(feature_frame)}')
feature_frame.head()

Feature frame rows: 134086


,client_hash_id,content_hash_id,gsc_impressions_mar,gsc_clicks_mar,avg_position_mar,ga4_pageviews_mar,ga4_engaged_sessions_mar,content_created_date,word_count,content_type,backlinks,gsc_impressions_feb,is_declining_proxy,pct_change_mar_vs_feb
0,client_157ffe4d4a595515,content_88b1daa0918ed139,276.0,2.0,5.428830,6.0,0.0,2026-02-13,3240,keyword article,0,160.0,0,0.725000
1,client_157ffe4d4a595515,content_88b4c2b2050326d0,1277.0,1.0,2.970158,1.0,0.0,2026-01-09,2573,comparison article,0,533.0,0,1.395872
2,client_157ffe4d4a595515,content_88c03e8eb0d7089b,726.0,2.0,8.169038,5.0,0.0,2026-01-19,2622,comparison article,0,49.0,0,13.816327
3,client_157ffe4d4a595515,content_88fa1a42b0ed7612,168.0,1.0,9.234822,1.0,0.0,2026-01-16,2553,comparison article,0,34.0,0,3.941176
4,client_157ffe4d4a595515,content_88fe52bf9d05da18,85.0,0.0,8.027095,NaN,NaN,2026-01-16,2599,comparison article,0,34.0,0,1.500000


### 3.5 The five features — one line each: "knowable at the decision moment because…"

- `gsc_impressions_mar` — total GSC impressions in March. Knowable at the decision moment because it's fully observed by the end of March, before any decision to review the page is made.
- `ctr_mar = gsc_clicks_mar / gsc_impressions_mar` — knowable at the decision moment for the same reason: derived entirely from fully-observed March data, no forward-looking window.
- `avg_position_mar` — average GSC position in March. Knowable at the decision moment because GSC position is measured day by day as the month happens, not retroactively.
- `content_age_days = days between content_created_date and 2026-03-31`. Knowable at the decision moment because `content_created_date` is always in the past relative to the review decision — you always know how old a page is.
- `ga4_engagement_rate_mar = ga4_engaged_sessions_mar / ga4_pageviews_mar`, only where GA4 was available all month. Knowable at the decision moment for the same reason as CTR — fully observed March data, restricted to clients with real GA4 coverage (filtered via `ga4_data_available IS TRUE` upstream, in 3.4).


In [7]:
import pandas as pd
import numpy as np

feature_frame['ctr_mar'] = feature_frame['gsc_clicks_mar'] / feature_frame['gsc_impressions_mar'].replace(0, np.nan)
feature_frame['content_age_days'] = (pd.Timestamp('2026-03-31') - pd.to_datetime(feature_frame['content_created_date'])).dt.days
feature_frame['ga4_engagement_rate_mar'] = feature_frame['ga4_engaged_sessions_mar'] / feature_frame['ga4_pageviews_mar'].replace(0, np.nan)

HONEST_FEATURES = ['gsc_impressions_mar', 'ctr_mar', 'avg_position_mar', 'content_age_days', 'ga4_engagement_rate_mar']

print(f'{len(HONEST_FEATURES)} features built:', HONEST_FEATURES)
feature_frame[HONEST_FEATURES + ['is_declining_proxy']].describe()

5 features built: ['gsc_impressions_mar', 'ctr_mar', 'avg_position_mar', 'content_age_days', 'ga4_engagement_rate_mar']


,gsc_impressions_mar,ctr_mar,avg_position_mar,content_age_days,ga4_engagement_rate_mar,is_declining_proxy
count,134086.000000,134086.000000,134086.000000,134086.000000,51830.000000,134086.000000
mean,1948.068896,0.004029,15.965394,209.778433,0.026015,0.201125
std,6127.873190,0.030342,16.638123,116.418893,0.102294,0.400843
min,1.000000,0.000000,0.000000,31.000000,0.000000,0.000000
25%,41.000000,0.000000,5.260967,104.000000,0.000000,0.000000
50%,270.000000,0.000000,8.983158,214.000000,0.000000,0.000000
75%,1436.000000,0.002356,20.761442,266.000000,0.000000,0.000000
max,617124.000000,1.000000,297.000000,494.000000,1.000000,1.000000


### 3.6 The trap — add one label-derived column on purpose

`pct_change_mar_vs_feb` is literally the quantity the label is thresholded from (see the SQL in 3.4). It should never be a real feature — but let's watch what happens if it sneaks in, exactly like `trend_direction`/`trend_pct` did in the starter CSV back in w02.

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import roc_auc_score

data = feature_frame.dropna(subset=HONEST_FEATURES + ['pct_change_mar_vs_feb', 'is_declining_proxy']).copy()
y = data['is_declining_proxy'].values
groups = data['client_hash_id'].values
gkf = GroupKFold(n_splits=5)

# --- WITH the leaked column ---
X_leaky = data[HONEST_FEATURES + ['pct_change_mar_vs_feb']].values
clf = LogisticRegression(max_iter=1000)
pred_leaky = cross_val_predict(clf, X_leaky, y, cv=gkf, groups=groups, method='predict_proba')[:, 1]
auc_leaky = roc_auc_score(y, pred_leaky)

print(f'AUC WITH the label-derived column (pct_change_mar_vs_feb) included: {auc_leaky:.3f}')
print('Expect this to sit right up near 1.0 -- that is the leak, not a good model.')

AUC WITH the label-derived column (pct_change_mar_vs_feb) included: 1.000
Expect this to sit right up near 1.0 -- that is the leak, not a good model.


In [9]:
# --- delete the leaked column, keep the honest number ---
X_honest = data[HONEST_FEATURES].values
pred_honest = cross_val_predict(clf, X_honest, y, cv=gkf, groups=groups, method='predict_proba')[:, 1]
auc_honest = roc_auc_score(y, pred_honest)

print(f'AUC WITHOUT the leaked column (5 honest features only): {auc_honest:.3f}')
print(f'Drop from the leaky number: {auc_leaky - auc_honest:.3f}')
print('This second number is the one I keep and report -- the first was never a real signal, just the label looking at itself.')

AUC WITHOUT the leaked column (5 honest features only): 0.631
Drop from the leaky number: 0.369
This second number is the one I keep and report -- the first was never a real signal, just the label looking at itself.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Answer

**Named limitation** — one month is a short, seasonally-confounded baseline. Comparing March to February alone can't distinguish a genuine content-quality decline from ordinary seasonality (e.g. a tax-prep page naturally drops every March regardless of content freshness) or a one-off SERP algorithm update that hit many pages at once, client-wide. A two-month comparison has no way to separate "this page got worse" from "everything moved together that month." A more honest version of this label would use a longer trailing baseline (e.g. trailing 3–6 months) or compare each page's change to its own client's median change that month, so a client-wide swing isn't mistaken for a page-specific signal.

**Also worth naming, briefly:**
- History depth is unbalanced across clients (`dim_clients.gsc_data_start` / `ga4_data_start` differ), so "March 2026" is not an equally-long history for every client — some clients may have very little prior data to compare against.
- GA4 rows before a client's `ga4_data_start` are zero-filled with `ga4_data_available = FALSE`; already filtered out above, but it's a trap for anyone querying this table without checking the flag first.
- `fact_content_query_90d` was deliberately excluded here (see Section 1) — that means query-level signals (rare-query share, anonymized-impressions share) are not available to this month's feature frame at all, only to later work that treats the sealed final month correctly.

**Scope limitation relative to Lane 2** — this notebook's label, `is_declining_proxy`, only tells me something changed for the worse; it can't tell me the *action* to take. A declining page could warrant a refresh, or pruning if it was never worth keeping in the first place — this label doesn't distinguish those. It also produces no signal at all for expansion, protection, or monitoring candidates, which are three of Lane 2's five target actions. Reason codes, confidence labels, and the full 5-way action assignment are out of scope for this data-contract week and belong to the baseline-building week (ML-07), built on top of this same feature set.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.